In [37]:
import optuna
from multiprocessing import Pool
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import Trial
import threading


import neuron
from neuron import h
import matplotlib.pyplot as plt
from neuron.units import ms, mV
import numpy as np
import pickle
import os
from functools import partial

from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy import signal
# Load custom modules
pth_mod = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod'
pth_rslts = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/new_cells_dev/cells'
import sys
sys.path.append('/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/new_cells_dev/utils')
from cell_builder_utils import l2_params, _get_dends, _get_pyr_soma, Cell
from iascone_mechs_utils import Mechs, get_iascone_params, Ih_condu_apical, distribute_gbar
from hay_mechs_utils import linear_gbar

In [3]:
target_median

{'RMP': np.float64(-71.57434844970703),
 'sag': np.float64(0.12074996220568815),
 'mean_isi': np.float64(97.37777777777778),
 'first_isi': np.float64(20.059999999999967),
 'upstroke_downstroke': np.float64(3.462917831744481),
 'ap_width': np.float64(0.0008600000000000829),
 'ap_threshold': np.float64(-47.23437690734863),
 'FR_slope': np.float64(0.06826516329253936),
 'tau': np.float64(23.524146358665664),
 'lambda_atten': 290,
 'lambda_upstroke': 121,
 'num_spikes': 5,
 'peak_heights': (np.float64(-18.554297323332722),
  np.float64(0.9944783563207278)),
 'prominences': (np.float64(38.654371188396), np.float64(65.04015610229789)),
 'widths': (np.float64(1189.7060372185097), np.float64(1904.4336505803876))}

In [ ]:
# define cell model
def _fit_len_const(x, lambda_val, A):
    return A * np.exp(-x / lambda_val)

def _fit_taum(t, tau, A, C):
    return A * (1 - np.exp(-t / tau)) + C

class L23PyramidalCellModel():
    def __init__(self, name='L23_pyr_cell'):
        self.pth_mod = pth_mod
        self.mech_class = Mechs(self.pth_mod)
        
        # Get default parameters
        self.section_params, self.end_pts, self.cell_tree = l2_params()
        self.section_names = list(self.end_pts.keys())
        
        # Voltage initialization
        self.v_init = {'soma': -68.85159703362191,
                        'basal_1': -68.9013799898162,
                        'basal_2': -69.04833422207797,
                        'basal_3': -69.04833422207797,
                        'apical_oblique': -68.67137672693937,
                        'apical_trunk': -68.69214528583284,
                        'apical_1': -67.42523827628489,
                        'apical_tuft': -65.21723613840336}
        
        # Hand-tuned mechanism parameters
        self.default_params = {'soma': {'NaTs2_t_32d': 20_400e-4*3.5,
                                'SKv3_1': 6_930e-4*4.5,
                                'Nap_Et2': 1e-4/2,
                                'Ca_HVA': 9.92e-04,
                                'Ca_LVAst': 34.3e-4,
                                
                                #'SK_E2': 0.099433/2*0.001,
                                'pas': 1.0 / 35000.0,
                                'Ih': 0.000080*0.5,
                                'Im': 8e-3},
                        'basal': {'NaTs2_t_32d': 0.008009*0.01,#partial(linear_gbar,0.008009/2,0,255),
                                  'SKv3_1': 0.000513*0.01,#partial(linear_gbar,0,6_930e-4*0.1,255),
                                  'Ih': 0.00008,
                                  'pas':  1.0 / 35000.0},
                        'dend': {'NaTa_t_32d': 0.008009*0.1,
                                'SKv3_1': 0.000513*0.1,
                                'Ca_HVA': partial(linear_gbar, 2.78e-5/2*1., 2.78e-5/2*1, 400, hotzone=[400, 600], hotzone_factor=1),
                                'Ca_LVAst': partial(linear_gbar, 93.5e-6, 93.5e-6*1, 400, hotzone=[400, 600], hotzone_factor=1),
                                'pas': 1.0 / 35000.0,
                                #'SK_E2': 0.099433/2*0.00001,
                                'Ih': {'gbar': 0.000080, 'offset': -0.869600, 'slope':2, 'factor_exp':1/110},
                                'Im': 8e-3*0},
                        'e_pas': -75}
        
        self.sections = None
        self.cell = None
        self.build_cell()

    # build cell
    def build_cell(self, params=None):
        """Build the cell with given parameters or defaults"""
        if params is None:
            params = self.default_params
            
        # Build sections
        self.sections = _get_dends(self.section_params, 'L2Pyr', self.section_names, 
                                  self.end_pts, v_init=self.v_init)
        self.sections['soma'] = _get_pyr_soma(self.section_params, 'L2Pyr', 
                                             self.end_pts['soma'], v_init=self.v_init['soma'])
        
        # Create cell
        self.cell = Cell(self.sections, self.cell_tree, cell_type='L2_pyramidal', gid=0)
        self.cell.build()
        
        # Add mechanisms
        self.cell = self.mech_class.add_mechs(self.cell, params)
        
        return self.cell
    
    # run protocol function
    def run_protocol(self, protocol, param_values=None):
        """Run a protocol and return responses"""
        # Update parameters
        self.build_cell(param_values)

        if protocol.name == 'FR_slope':
            return self.run_FR(protocol)
        elif protocol.name == 'attenuation':
            return self.run_attenuation(protocol)
        else:
            return self.run_stim(protocol)
        
    # template simulation function
    def run_stim(self, protocol):
        """Run dendritic stimulation protocol"""
        # Implementation of dendritic stimulation
        responses = {}
        
        # Set up stimulation
        stim = h.IClamp(self.cell._nrn_sections[protocol.section](protocol.seg))
        stim.delay = protocol.stim_delay
        stim.dur = protocol.stim_dur
        stim.amp = protocol.stim_amp
        
        # Set up recording
        t = h.Vector().record(h._ref_t)
        v = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_v)
        ca = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_cai)
        
        # Run simulation
        h.celsius = 37
        h.finitialize()
        h.continuerun(protocol.total_dur * ms)
        
        responses['time'] = np.array(t)
        responses[f'{protocol.section}_v'] = np.array(v)
        responses[f'{protocol.section}_ca'] = np.array(ca)
    
        return responses
    
    def reset_cell_state(self):
        """Reset the cell to its initial state"""
        # Reset membrane potential
        for sec_name, v in self.v_init.items():
            for seg in self.cell._nrn_sections[sec_name]:
                seg.v = v
                
        # Reset calcium concentrations if needed
        for sec_name, section in self.sections.items():
            for seg in self.cell._nrn_sections[sec_name]:
                if hasattr(seg, 'cai'):
                    seg.cai = 5e-5
    
    def run_FR(self, protocol):

        responses = {}
        
        self.current_stim = None
        for a, amp in enumerate(protocol.stim_amp):

            if self.current_stim:
                self.current_stim = None
                
            # Set up stimulation with current amplitude
            stim = h.IClamp(self.cell._nrn_sections[protocol.section](protocol.seg))
            stim.delay = protocol.stim_delay
            stim.dur = protocol.stim_dur
            stim.amp = amp
            self.current_stim = stim

            # Set up recording
            t = h.Vector().record(h._ref_t)
            v = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_v)

            self.reset_cell_state()
            
            # Run simulation
            h.celsius = 37
            h.dt = 0.01
            h.finitialize()
            h.continuerun(protocol.total_dur * ms)

            

            responses[amp] = {
                'time': np.array(t),
                'v': np.array(v)
            }

        return responses
    
    def run_attenuation(self, protocol):

        apcl_names = ['soma', 'apical_trunk', 'apical_1', 'apical_tuft']
        sections_minus_ref = [section for section in apcl_names if section != protocol.section]

        responses = {}
        # Set up stimulation with current amplitude
        stim = h.IClamp(self.cell._nrn_sections[protocol.section](protocol.seg))
        stim.delay = protocol.stim_delay
        stim.dur = protocol.stim_dur
        stim.amp = protocol.stim_amp
        self.current_stim = stim

        # Set up recording
        t = h.Vector().record(h._ref_t)
        ref_v = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_v)
        distance = []
        v_dend = []
        for s, section in enumerate(sections_minus_ref):
            for sg, seg in enumerate(self.cell._nrn_sections[section]):
                v_dend.append(h.Vector().record(self.cell._nrn_sections[section](seg.x)._ref_v))
                distance.append(h.distance(seg))

        h.celsius = 37
        h.finitialize()
        h.continuerun(250 * ms)
    
        c = 0
        for s, section in enumerate(sections_minus_ref):
            for sg, seg in enumerate(self.cell._nrn_sections[section]):
                responses[f'{section}_{seg.x}_v'] = np.array(v_dend[c])
                c+=1

        responses['time'] = np.array(t)
        responses['soma_v'] = np.array(ref_v)
        responses['distance'] = np.array(distance)

        return responses
    
    # Attenuation
    def calculate_attenuation(self, responses):
        """ Get attenuation from soma to tuft"""

        pk = find_peaks(responses['soma_v'], height=0)[0]
        
        apcl_names = ['soma', 'apical_trunk', 'apical_1', 'apical_tuft']
        sections_minus_ref = [section for section in apcl_names if section != 'soma']

        atten = []
        upstroke = []
        if len(pk) > 0:
            amp_soma = np.max(responses['soma_v']) - np.min(responses['soma_v'])

            for s, section in enumerate(sections_minus_ref):
                for sg, seg in enumerate(self.cell._nrn_sections[section]):
                    amp_dend = np.max(responses[f'{section}_{seg.x}_v']) - np.min(responses[f'{section}_{seg.x}_v'])
                    atten.append(np.array(amp_dend) / np.array(amp_soma) * 100)

                    # upstroke per ms (Gidon et al., 2020 Fig S5 A)
                    upstroke.append(np.max(np.diff(responses[f'{section}_{seg.x}_v']))/np.diff(responses['time'])[0])
                
        else:
            atten = np.zeros_like(responses['distance'])
            upstroke = np.zeros_like(responses['distance'])

        popt, pcov = curve_fit(_fit_len_const, np.array(responses['distance']), np.array(atten), p0=[100, atten[0] ])
        lambda_atten = popt[0]
        popt, pcov = curve_fit(_fit_len_const, np.array(responses['distance']), np.array(upstroke), p0=[100, upstroke[0] ])
        lambda_upstroke = popt[0]

        return lambda_atten, lambda_upstroke
    
    # RMP
    def calculate_RMP(self, responses, protocol=None):
        """ Get RMP """
        return responses['soma_v'][-3]
    
    def calculate_isi(self, responses, protocol=None):
        """Calculate inter-spike intervals (ISI) from the voltage response"""
        
        # get APs
        aps = find_peaks(responses['soma_v'],height=0)[0]

        first_isi = np.diff(responses['time'][aps])[0] if len(aps) > 1 else np.nan
        mean_isi = np.mean(np.diff(responses['time'][aps])) if len(aps) > 1 else np.nan

        return first_isi, mean_isi

    def calculate_sag(self, responses, protocol=None):
        """Calculate the sag ratio from the voltage response"""
        min_membrane_pot = np.min(responses['soma_v'])
        steady_idx = np.argmin(np.abs(responses['time'] - 180))      # approximately steady state
        steady_state = responses['soma_v'][steady_idx]   
        return (min_membrane_pot-steady_state)/min_membrane_pot
    
    def calculate_taum(self, responses, protocol):
        """Calculate the time constant of the membrane"""
        
        start_idx = np.where(responses['time'] >= protocol.stim_delay)[0][0]
        end_idx = np.where(responses['time'] >= protocol.stim_delay + protocol.stim_dur)[0][0]

        # get rise of response
        t_rise = responses['time'][start_idx:end_idx] - protocol.stim_delay
        v_rise = responses['soma_v'][start_idx:end_idx] 

        # change from RMP
        delta_v = np.mean(v_rise) - np.mean(responses['soma_v'][:start_idx])

        # Fit the exponential function to the data
        popt, pcov = curve_fit(_fit_taum, t_rise, v_rise, p0=[10, abs(delta_v), v_rise[0]])
        return popt[0] # This is the fitted time constant in ms
    
    def calculate_ap(self, responses, protocol=None):
        """Calculate action potential measures"""
        # Get APs
        ap, properties = find_peaks(responses['soma_v'], height=0)
        
        if len(ap) == 0:
            return 0, 0, 0, 0
        elif len(ap) > 1:
            return np.inf, np.inf, np.inf, np.inf
        
        # peak

        try:
            ap_peak = properties['peak_heights'][0]
            
            # smooth AP and find threshold
            smooth_v = signal.savgol_filter(np.diff(responses['soma_v']), 101, 3)

            ap_thresh_idx = np.where(smooth_v/np.diff(responses['soma_v']) > 20)[0][0]+1

            ap_thresh = responses['soma_v'][ap_thresh_idx]        
        

            # AP width
            ap_trough = np.argmin(responses['soma_v'][np.argmax(responses['soma_v']):])

            half_max = np.min(responses['soma_v']) + (np.max(responses['soma_v'])-np.min(responses['soma_v']))/2
            idx1 = np.argmin(np.abs(responses['soma_v'][ap_thresh_idx:int(ap)]-half_max))
            idx2 = np.argmin(np.abs(responses['soma_v'][int(ap):int(ap_trough+np.argmax(responses['soma_v']))]-half_max))
            
            ap_width = responses['time'][idx2+int(ap)]-responses['time'][idx1+ap_thresh_idx]

            # upstroke: mean of dV/dt between AP threshold and AP peak
            upstroke = np.mean(np.gradient(responses['soma_v'][ap_thresh_idx:int(ap)]))

            # downstroke: minimum value of dV/dt between AP peak and trough
            pk, properties = find_peaks(smooth_v[int(ap)+1:], height=0)
            ap_trough = int(ap+1+pk[0])
            downstroke = np.mean(np.gradient(responses['soma_v'][int(ap):ap_trough]))
            return ap_peak, ap_thresh, ap_width, np.abs(upstroke/downstroke)
        except(IndexError, KeyError, ValueError) as e:
            print('trial failed')
            return 0, 0, 0, 0
        
        
        return ap_peak, ap_thresh, ap_width, np.abs(upstroke/downstroke)
    
    def calculate_fr(self, responses, protocol):
        """Calculate firing rate slope"""
        fr = np.zeros(len(responses))
        for a, amp in enumerate(responses.keys()):
            pk = find_peaks(responses[amp]['v'], height=0)

            fr[a] = len(pk[0])

        fr_slope = np.polyfit(protocol.stim_amp, fr, 1)[0]*0.001

        return fr_slope
    
    def calculate_ca(self, responses, protocol=None):
        """Calculate calcium spike measures"""

        # get ca spikes from dendrite
        pks, properties = find_peaks(responses['dendrite_v'], height=-40, prominence=5, width=5)

        prop_ca_spikes = {}
        prop_ca_spikes['num_spikes'] = len(pks)
        prop_ca_spikes['peak_heights'] = np.mean(properties['peak_heights'])
        prop_ca_spikes['prominences'] = np.mean(properties['prominences']), max(properties['prominences'])
        prop_ca_spikes['widths'] = np.mean(properties['widths'])

        return prop_ca_spikes
    
    

class StimProtocol:
    def __init__(self, name, stim_delay, stim_dur, stim_amp, total_dur, section, seg):
        self.name = name
        self.stim_delay = stim_delay
        self.stim_dur = stim_dur
        self.stim_amp = stim_amp
        self.total_dur = total_dur
        self.section = section
        self.seg = seg



In [ ]:
def _get_protocols():

    RMP_protocol = StimProtocol(
        name='RMP',
        stim_delay=0 * ms,
        stim_dur=0 * ms,
        stim_amp=0,
        total_dur=200 * ms,
        section='soma',
        seg=0.5
    )

    isi_protocol = StimProtocol(
        name='isi',
        stim_delay=20 * ms,
        stim_dur=1000 * ms,
        stim_amp=1.,
        total_dur=1020 * ms,
        section='soma',
        seg=0.5
    )

    sag_protocol = StimProtocol(
        name='sag',
        stim_delay=20*ms,
        stim_dur=200*ms,
        stim_amp=-1,
        total_dur=220*ms,
        section='soma',
        seg=0.5
    )

    taum_protocol = StimProtocol(
        name='tau',
        stim_delay=30 * ms,
        stim_dur=100 * ms,
        stim_amp=-0.15,
        total_dur=200 * ms,
        section='soma',
        seg=0.5
    )

    ap_protocol = StimProtocol(
        name='ap',
        stim_delay=20 * ms,
        stim_dur=3 * ms,
        stim_amp=2,
        total_dur=60 * ms,
        section='soma',
        seg=0.5
    )

    fr_protocol = StimProtocol(
        name='FR_slope',
        stim_amp = np.linspace(.2, 2., 10),
        stim_delay = 20,
        stim_dur = 1000,
        total_dur = 1020,
        section='soma',
        seg=0.5
    )

    atten_protocol = StimProtocol(
        name='attenuation',
        stim_delay=20,
        stim_dur=3,
        stim_amp=1.5,
        total_dur=200,
        section='soma',
        seg=0.5
    )

    ca_spike_protocol = StimProtocol(
        name='ca_spike',
        stim_delay = 100,
        stim_dur = 1000,
        stim_amp = 0.25,
        total_dur=1000,
        section='apical_tuft',
        seg=0.5
    )

    return {
        'RMP': (RMP_protocol, partial(l23cell.calculate_RMP)),
        'isi': (isi_protocol, partial(l23cell.calculate_isi)),
        'sag': (sag_protocol, partial(l23cell.calculate_sag)),
        'tau': (taum_protocol, partial(l23cell.calculate_taum)),
        'ap': (ap_protocol, partial(l23cell.calculate_ap)),
        'FR_slope': (fr_protocol, partial(l23cell.calculate_fr)),
        'atten': (atten_protocol, partial(l23cell.calculate_attenuation)),
        'ca_spike': (ca_spike_protocol, partial(l23cell.calculate_ca))
    }


In [25]:
l23cell = L23PyramidalCellModel()

protocols = _get_protocols()
protocols

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


{'RMP': (<__main__.StimProtocol at 0x1768d8760>,
  functools.partial(<bound method L23PyramidalCellModel.calculate_RMP of <__main__.L23PyramidalCellModel object at 0x1768d84c0>>)),
 'isi': (<__main__.StimProtocol at 0x1768d8250>,
  functools.partial(<bound method L23PyramidalCellModel.calculate_isi of <__main__.L23PyramidalCellModel object at 0x1768d84c0>>)),
 'sag': (<__main__.StimProtocol at 0x1768d8040>,
  functools.partial(<bound method L23PyramidalCellModel.calculate_sag of <__main__.L23PyramidalCellModel object at 0x1768d84c0>>)),
 'tau': (<__main__.StimProtocol at 0x176910670>,
  functools.partial(<bound method L23PyramidalCellModel.calculate_taum of <__main__.L23PyramidalCellModel object at 0x1768d84c0>>)),
 'ap': (<__main__.StimProtocol at 0x176910ee0>,
  functools.partial(<bound method L23PyramidalCellModel.calculate_ap of <__main__.L23PyramidalCellModel object at 0x1768d84c0>>)),
 'FR_slope': (<__main__.StimProtocol at 0x176910910>,
  functools.partial(<bound method L23Pyram

In [ ]:
def objective(trial):

    print(f"Running trial {trial.number=} in {threading.current_thread().name}")
    l23cell = L23PyramidalCellModel()

    protocols = _get_protocols()

    ca_hva_gbar = trial.suggest_float('ca_hva_gbar', 0, 2e-3)
    ca_hva_slope = trial.suggest_float('ca_hva_slope', 0, 10)
    ca_hva_hotfact = trial.suggest_float('ca_hva_hotfact', 0, 20)

    ca_lva_gbar = trial.suggest_float('ca_lva_gbar', 0, 2e-3)
    ca_lva_slope = trial.suggest_float('ca_lva_slope', 0, 10)
    ca_lva_hotfact = trial.suggest_float('ca_lva_hotfact', 0, 20)
    
    suggest_params = {'soma': {'NaTs2_t_32d': trial.suggest_float('NaTs2_t_32d', 0, 8),
                                    'SKv3_1': trial.suggest_float('SKv3_1', 0, 5),
                                    'Nap_Et2': trial.suggest_float('Nap_Et2', 0, 1),
                                    'Ca_HVA': trial.suggest_float('Ca_HVA', 0, 1),
                                    'Ca_LVAst': trial.suggest_float('Ca_LVAst', 0, 1),
                                    'pas': trial.suggest_float('pas', 0, 1e-3),
                                    'Ih': trial.suggest_float('Ih', 0, 1e-4),
                                    'Im': trial.suggest_float('Im', 0, 1e-2)},
                  'basal': {'NaTs2_t_32d': trial.suggest_float('NaTs2_t_32d_basal', 0, 2e-3),
                                    'SKv3_1': trial.suggest_float('SKv3_1_basal', 0, 2e-3),
                                    'Ih': trial.suggest_float('Ih_basal', 0, 1e-3),
                                    'pas':  trial.suggest_float('pas_basal', 0, 1/9000)},
                            'dend': {'NaTa_t_32d': trial.suggest_float('NaTa_t_32d_apical', 0, 1e-2),
                                    'SKv3_1': trial.suggest_float('SKv3_1_apical', 0, 1e-2),
                                    'Ca_HVA': partial(linear_gbar, ca_hva_gbar, ca_hva_gbar*ca_hva_slope, 400, hotzone=[400, 600], hotzone_factor=ca_hva_hotfact),
                                    'Ca_LVAst': partial(linear_gbar, ca_lva_gbar, ca_lva_gbar*ca_lva_slope, 400, hotzone=[400, 600], hotzone_factor=ca_lva_hotfact),
                                    'pas': trial.suggest_float('pas_apical', 0, 1/9000.0),
                                    'SK_E2': trial.suggest_float('SK_E2_apical', 0, 1e-4),
                                    'Ih': {'gbar': trial.suggest_float('Ih_gbar', 0, 1e-2), 'offset': -0.869600, 'slope':2, 'factor_exp':1/110},
                                    'Im': trial.suggest_float('Im_apical', 0, 1e-2)},
                            'e_pas': -75}
    
    # work in stages like Hay et al
    one_opt = ['RMP', 'sag', 'tau', 'FR_slope']

    loss = 0
    for opt in one_opt:
        responses = l23cell.run_protocol(protocols[opt][0], suggest_params)

        measure = protocols[opt][1](responses, protocols[opt][0])
        loss += (measure - target_median[opt]) ** 2

    # action potential measures

    responses = l23cell.run_protocol(protocols['ap'][0], suggest_params)
    ap_peak, ap_thresh, ap_width, up_down = protocols['ap'][1](responses, protocols['ap'][1])
    loss += (ap_thresh-target_median['ap_threshold'])**2
    loss += (ap_width-target_median['ap_width'])**2
    loss += (up_down-target_median['upstroke_downstroke'])

    # isi measures
    responses = l23cell.run_protocol(protocols['isi'][0], suggest_params)
    first_isi, mean_isi = protocols['isi'][1](responses, protocols['isi'][0])
    loss += (first_isi-target_median['first_isi'])**2
    loss += (mean_isi-target_median['mean_isi'])**2

    return loss

In [47]:
l23cell = L23PyramidalCellModel()

protocols = _get_protocols()
responses = l23cell.run_protocol(protocols['FR_slope'][0])
fr = protocols['FR_slope'][1](responses, protocols['FR_slope'][0])
fr

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


np.float64(0.062303030303030305)

In [48]:
study = optuna.create_study(direction='minimize', storage=JournalStorage(JournalFileBackend(file_path="./journal.log")))
study.optimize(objective, n_trials=10)

[I 2025-08-23 18:44:04,884] A new study created in Journal with name: no-name-9df717cf-f125-49b4-a4d9-d2f0ccdeea58


Running trial trial.number=0 in MainThread
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-23 18:44:05,865] Trial 0 finished with value: 4435.913890142301 and parameters: {'ca_hva_gbar': 0.0005229892480109218, 'ca_hva_slope': 4.954978579686137, 'ca_hva_hotfact': 8.150936017451684, 'ca_lva_gbar': 0.0007521291836776793, 'ca_lva_slope': 0.9667346197229898, 'ca_lva_hotfact': 1.3674449637044273, 'NaTs2_t_32d': 0.6333909835763007, 'SKv3_1': 2.8166247665733195, 'Nap_Et2': 0.41836202880760887, 'Ca_HVA': 0.15957167096046299, 'Ca_LVAst': 0.4173401121262371, 'pas': 0.0007775240580985013, 'Ih': 6.183537014526265e-05, 'Im': 0.00016392068968248852, 'NaTs2_t_32d_basal': 0.0015590471505929774, 'SKv3_1_basal': 0.0010291775036807675, 'Ih_basal': 0.0008981970497696133, 'pas_basal': 0.00010493472145393744, 'NaTa_t_32d_apical': 0.004267944934589269, 'SKv3_1_apical': 0.0069681030143383125, 'pas_apical': 8.518829084790916e-05, 'SK_E2_apical': 6.663456315177447e-05, 'Ih_gbar': 0.004792617956365399, 'Im_apical': 0.009881276138445654}. Best is trial 0 with value: 4435.913890142301.


Running trial trial.number=1 in MainThread
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-23 18:44:06,829] Trial 1 finished with value: 7360.819584206523 and parameters: {'ca_hva_gbar': 0.0011678730160098168, 'ca_hva_slope': 5.1342634461973695, 'ca_hva_hotfact': 14.832612661151694, 'ca_lva_gbar': 0.0017918888404619133, 'ca_lva_slope': 7.733911580720064, 'ca_lva_hotfact': 16.544375069669147, 'NaTs2_t_32d': 4.058595777610769, 'SKv3_1': 0.45456520743558104, 'Nap_Et2': 0.7822631025711447, 'Ca_HVA': 0.1730235487982319, 'Ca_LVAst': 0.9448166999835078, 'pas': 0.0009082825250113475, 'Ih': 6.74417799249203e-05, 'Im': 0.008113226277109338, 'NaTs2_t_32d_basal': 0.001077986872833202, 'SKv3_1_basal': 0.000807087209490821, 'Ih_basal': 0.000361583702221669, 'pas_basal': 7.838644124388883e-05, 'NaTa_t_32d_apical': 0.004090820646424326, 'SKv3_1_apical': 0.007020237961889595, 'pas_apical': 4.747045718197587e-05, 'SK_E2_apical': 4.530404038526704e-05, 'Ih_gbar': 0.004507354330218761, 'Im_apical': 0.004587935295793178}. Best is trial 0 with value: 4435.913890142301.


Running trial trial.number=2 in MainThread
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


... (11 outputs left out)

Running trial trial.number=8 in MainThread
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-23 18:44:13,557] Trial 8 finished with value: 5622.510154870736 and parameters: {'ca_hva_gbar': 0.0004228141533082939, 'ca_hva_slope': 8.34230123108809, 'ca_hva_hotfact': 14.283735132832362, 'ca_lva_gbar': 0.0010126990667090833, 'ca_lva_slope': 1.8418936050184653, 'ca_lva_hotfact': 14.892813134700226, 'NaTs2_t_32d': 2.545388580328483, 'SKv3_1': 1.5225916668763757, 'Nap_Et2': 0.6008405488532236, 'Ca_HVA': 0.8534057988676478, 'Ca_LVAst': 0.9501272844522645, 'pas': 0.0007351080280598135, 'Ih': 5.53582013639018e-06, 'Im': 0.005559034792208728, 'NaTs2_t_32d_basal': 0.0006351234439718301, 'SKv3_1_basal': 0.0013482370388994064, 'Ih_basal': 0.0006133164049365552, 'pas_basal': 3.500151119673076e-05, 'NaTa_t_32d_apical': 0.00012575597609715717, 'SKv3_1_apical': 0.004602418380742814, 'pas_apical': 0.00010536386819750445, 'SK_E2_apical': 2.6762922744222806e-05, 'Ih_gbar': 0.0037759686770424484, 'Im_apical': 0.0061099523895515855}. Best is trial 4 with value: 4180.450324848985.


Running trial trial.number=9 in MainThread
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-23 18:44:14,498] Trial 9 finished with value: 4204.364193394707 and parameters: {'ca_hva_gbar': 2.923576608595102e-06, 'ca_hva_slope': 2.404577168280584, 'ca_hva_hotfact': 10.81447558843765, 'ca_lva_gbar': 0.0005169946885039629, 'ca_lva_slope': 4.734227139842141, 'ca_lva_hotfact': 19.96874370209034, 'NaTs2_t_32d': 7.850719611829751, 'SKv3_1': 3.1225397440681633, 'Nap_Et2': 0.2698877015911122, 'Ca_HVA': 0.5699737173284649, 'Ca_LVAst': 0.6744753764255984, 'pas': 0.0009452138089374668, 'Ih': 4.3487115516531466e-05, 'Im': 0.0016848114995044595, 'NaTs2_t_32d_basal': 0.001676444301099968, 'SKv3_1_basal': 0.0005358008300607504, 'Ih_basal': 0.00015798671321260417, 'pas_basal': 4.148732505095365e-05, 'NaTa_t_32d_apical': 0.007777382309625975, 'SKv3_1_apical': 0.007786532669240667, 'pas_apical': 8.273200014073021e-05, 'SK_E2_apical': 6.477713208252021e-05, 'Ih_gbar': 0.00482446952629348, 'Im_apical': 0.0031343113674773848}. Best is trial 4 with value: 4180.450324848985.


In [ ]:
with open(os.path.join(pth_rslts, 'optuna_l23soma_optim.pkl'), 'wb') as f:
    pickle.dump(study.best_trial, f)

In [50]:
with open(os.path.join(pth_rslts, 'optuna_l23soma_optim.pkl'), 'rb') as f:
    test = pickle.load(f)

test

FrozenTrial(number=49, state=TrialState.COMPLETE, values=[7529.698993850515], datetime_start=datetime.datetime(2025, 8, 23, 18, 48, 41, 164535), datetime_complete=datetime.datetime(2025, 8, 23, 18, 48, 58, 966830), params={'ca_hva_gbar': 0.0018714375827762102, 'ca_hva_slope': 6.234264041268281, 'ca_hva_hotfact': 0.2604056135880395, 'ca_lva_gbar': 0.000581095891383111, 'ca_lva_slope': 4.826575215433651, 'ca_lva_hotfact': 6.535873636770373, 'NaTs2_t_32d': 4.460541276275606, 'SKv3_1': 1.8096983564247826, 'Nap_Et2': 0.003892835589297816, 'Ca_HVA': 0.034677543533300575, 'Ca_LVAst': 0.053352571217001654, 'pas': 0.0003092548763253591, 'Ih': 6.372572966341314e-05, 'Im': 0.003826056927055981, 'NaTs2_t_32d_basal': 0.0013142909217018803, 'SKv3_1_basal': 7.974188853555983e-05, 'Ih_basal': 0.0004526878405087649, 'pas_basal': 7.395803302221915e-05, 'NaTa_t_32d_apical': 0.003658697765704771, 'SKv3_1_apical': 0.007477051177212964, 'pas_apical': 3.1582198778871954e-05, 'SK_E2_apical': 4.00837762135657e

In [ ]:
study = optuna.create_study(study_name='l23_soma_optimization', direction='minimize', storage=JournalStorage(JournalFileBackend(file_path="./journal.log")), load_if_exists=True)
study.optimize(objective, n_trials=10)

if __name__ == "__main__":
    with Pool(processes=10) as pool:
        pool.map(run_optimization, range(12))